# Card datasets inventory

**Goal.** One table describing every table and view in the card / transaction databases we hold, so we can decide which to use and what to suggest to jjsinger and cwang.

**Scope.** Every database whose name starts with `EARNEST`, `FABLE`, `SHARE_CE`, `SHARE_EARNEST`, `SHARE_FABLE` or `SHARE_FACTEUS`.

**For each table / view**
- Who provides it, what kind of object it is, and its grain (row-level transactions, aggregate, reference, delivery metadata).
- When the data begins and when it runs until (`MIN` / `MAX` of its main date column).
- Whether it carries a transaction / billing description.
- Distinct transaction ids dated December 2025 (row-level tables only), to compare sizes.

**How it runs.** Metadata comes from each database's `INFORMATION_SCHEMA`. The data probes run one query per table, cached in `results/probes.csv`, so a rerun only redoes what is missing (set `RETRY_FAILED = True` to retry timeouts and errors). Databases that are exact copies of another are not probed again; they point at the copy that was (`mirror_of`).

Follow-up (not in this notebook yet): which tables are subsets of others, and which to use for a given analysis.

In [1]:
import os, re, time, threading, datetime as dt
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
import snowflake.connector

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 500)

CONN_NAME = os.getenv("SNOWFLAKE_CONNECTION", "default")
WAREHOUSE = os.getenv("CARD_INV_WAREHOUSE")             # e.g. LARGE for the heavy December counts
CONN_KW = {"warehouse": WAREHOUSE} if WAREHOUSE else {}
conn = snowflake.connector.connect(connection_name=CONN_NAME, **CONN_KW)

def q(sql: str, cur=None) -> pd.DataFrame:
    cur = cur or conn.cursor()
    cur.execute(sql)
    df = cur.fetch_pandas_all()
    df.columns = [c.lower() for c in df.columns]
    return df

HERE = os.path.abspath("")
RES = os.path.join(HERE, "results")
os.makedirs(RES, exist_ok=True)

PREFIXES = ("EARNEST", "FABLE", "SHARE_CE", "SHARE_EARNEST", "SHARE_FABLE", "SHARE_FACTEUS")
TODAY = dt.date.today()
DEC_FROM, DEC_TO = "2025-12-01", "2026-01-01"   # December 2025, end exclusive
MINMAX_TIMEOUT_S = 300                          # per query
DEC_TIMEOUT_S = int(os.getenv("CARD_INV_DEC_TIMEOUT_S", "1800"))
THREADS = 6
RETRY_FAILED = os.getenv("CARD_INV_RETRY_FAILED") == "1"   # retry cached timeouts / errors
DRY_RUN = os.getenv("CARD_INV_DRY_RUN") == "1"  # classify only, skip the data probes

## 1. Databases

`origin` is the provider account and share for imported (shared) databases. `available = False` means Snowflake reports the share as no longer available (the provider revoked it).

In [2]:
cur = conn.cursor()
cur.execute("SHOW DATABASES")
cols = [d[0] for d in cur.description]
dbs = pd.DataFrame(cur.fetchall(), columns=cols)
dbs = dbs[dbs["name"].str.upper().str.startswith(PREFIXES)][["name", "kind", "origin", "created_on", "owner"]].reset_index(drop=True)

def provider(db: str) -> str:
    rules = [
        (r"^EARNEST$", "SRS internal: Earnest-derived ETL (ticker extracts, ad hoc)"),
        (r"^FABLE$", "SRS internal: Fable-derived aggregates"),
        (r"^SHARE_CE_SPEND_201", "Consumer Edge: Yodlee card & bank feeds (201)"),
        (r"^SHARE_CE_SPEND_279", "Consumer Edge: Earnest spend 279 (legacy schema)"),
        (r"^SHARE_EARNEST_279$", "Earnest spend 279 (DIRECT share; same provider account as the CE shares)"),
        (r"^SHARE_CE_PRIME_ENRICHED_.*_2822", "Consumer Edge: Prime Enriched, new Earnest schema (2822)"),
        (r"^SHARE_CE_1[34]15_DAILY_SIGNAL", "Consumer Edge Transact: daily signal (1315 / 1415)"),
        (r"^SHARE_CE_TRANSACT_USA", "Consumer Edge Transact USA: aggregates"),
        (r"^SHARE_CE_HEALTHCARE", "Consumer Edge: healthcare claims (not card)"),
        (r"^SHARE_EARNEST_", "Earnest: other share"),
        (r"^SHARE_FABLE", "Fable: European card panels"),
        (r"^SHARE_FACTEUS", "Facteus"),
    ]
    return next((label for pat, label in rules if re.search(pat, db)), "other")

dbs["provider"] = dbs["name"].map(provider)

def available(db):
    try:
        conn.cursor().execute(f"SELECT 1 FROM {db}.INFORMATION_SCHEMA.SCHEMATA LIMIT 1")
        return True
    except Exception:
        return False

dbs["available"] = dbs["name"].map(available)
dbs.sort_values(["provider", "name"])

,name,kind,origin,created_on,owner,provider,available
14,SHARE_CE_TRANSACT_USA_COHORT,IMPORTED DATABASE,DMOEKXS.WIB77105.CE_TRANSACT_USA_COHORT,2025-10-28 13:58:59.009000-07:00,ACCOUNTADMIN,Consumer Edge Transact USA: aggregates,True
15,SHARE_CE_TRANSACT_USA_DAILY_SIGNAL,IMPORTED DATABASE,DMOEKXS.WIB77105.CE_TRANSACT_USA_DAILY_SIGNAL,2025-10-28 13:59:27.694000-07:00,ACCOUNTADMIN,Consumer Edge Transact USA: aggregates,True
16,SHARE_CE_TRANSACT_USA_GROUND_TRUTH,IMPORTED DATABASE,DMOEKXS.WIB77105.CE_TRANSACT_USA_GROUND_TRUTH,2025-10-28 13:59:38.862000-07:00,ACCOUNTADMIN,Consumer Edge Transact USA: aggregates,True
2,SHARE_CE_1315_DAILY_SIGNAL,IMPORTED DATABASE,XW17098.PW55959.CE_1315_DAILY_SIGNAL,2026-08-14 13:20:22.822000-07:00,ACCOUNTADMIN,Consumer Edge Transact: daily signal (1315 / 1415),True
3,SHARE_CE_1415_DAILY_SIGNAL,IMPORTED DATABASE,XW17098.PW55959.CE_1415_DAILY_SIGNAL,2026-08-14 13:20:02.510000-07:00,ACCOUNTADMIN,Consumer Edge Transact: daily signal (1315 / 1415),True
11,SHARE_CE_SPEND_279_FINANCIAL_AGGREGATE,IMPORTED DATABASE,XW17098.PW55959.SPEND_279_FINANCIAL_AGGREGATE,2026-07-28 17:52:48.681000-07:00,ACCOUNTADMIN,Consumer Edge: Earnest spend 279 (legacy schema),True
12,SHARE_CE_SPEND_279_FINANCIAL_PRIME_ENRICHED,IMPORTED DATABASE,XW17098.PW55959.SPEND_279_FINANCIAL_PRIME_ENRICHED,2026-07-28 17:53:56.120000-07:00,ACCOUNTADMIN,Consumer Edge: Earnest spend 279 (legacy schema),True
13,SHARE_CE_SPEND_279_FINANCIAL_PRIME_ENRICHED_TAU,IMPORTED DATABASE,XW17098.PW55959.SPEND_279_FINANCIAL_PRIME_ENRICHED_TAU,2026-07-28 17:38:05.212000-07:00,ACCOUNTADMIN,Consumer Edge: Earnest spend 279 (legacy schema),True
8,SHARE_CE_PRIME_ENRICHED_OMEGA_2822,IMPORTED DATABASE,XW17098.PW55959.PRIME_ENRICHED_OMEGA_2822,2026-07-28 17:36:12.111000-07:00,ACCOUNTADMIN,"Consumer Edge: Prime Enriched, new Earnest schema (2822)",True
9,SHARE_CE_PRIME_ENRICHED_TAU_2822,IMPORTED DATABASE,XW17098.PW55959.PRIME_ENRICHED_TAU_2822,2026-07-28 17:36:31.586000-07:00,ACCOUNTADMIN,"Consumer Edge: Prime Enriched, new Earnest schema (2822)",True


## 2. Tables, views and columns

From `INFORMATION_SCHEMA.TABLES` and `.COLUMNS` of every available database. `row_count` and `bytes` are Snowflake metadata (NULL for views). `last_altered` is the last time the object changed, a freshness proxy that needs no scan.

In [3]:
tabs, colz = [], []
for db in dbs.loc[dbs.available, "name"]:
    tabs.append(q(f"""SELECT table_catalog, table_schema, table_name, table_type, row_count, bytes,
                             clustering_key, created, last_altered
                      FROM {db}.INFORMATION_SCHEMA.TABLES
                      WHERE table_schema <> 'INFORMATION_SCHEMA'"""))
    colz.append(q(f"""SELECT table_catalog, table_schema, table_name, column_name, data_type, ordinal_position
                      FROM {db}.INFORMATION_SCHEMA.COLUMNS
                      WHERE table_schema <> 'INFORMATION_SCHEMA'"""))
objs = pd.concat(tabs, ignore_index=True)
objs["row_count"] = pd.to_numeric(objs.row_count, errors="coerce")   # Decimal from Snowflake
objs["bytes"] = pd.to_numeric(objs.bytes, errors="coerce")
cols = pd.concat(colz, ignore_index=True)
objs["fq"] = objs.table_catalog + "." + objs.table_schema + "." + objs.table_name
cols["fq"] = cols.table_catalog + "." + cols.table_schema + "." + cols.table_name
print(f"{len(objs):,} objects, {len(cols):,} columns")
objs.pivot_table(index="table_catalog", columns="table_type", values="table_name", aggfunc="count", fill_value=0)

3,404 objects, 50,681 columns


table_type,BASE TABLE,EXTERNAL TABLE,VIEW
table_catalog,,,
EARNEST,286,0,1
FABLE,19,0,0
SHARE_CE_1315_DAILY_SIGNAL,52,0,0
SHARE_CE_1415_DAILY_SIGNAL,31,0,0
SHARE_CE_PRIME_ENRICHED_OMEGA_2822,43,0,0
SHARE_CE_PRIME_ENRICHED_TAU_2822,43,0,0
SHARE_CE_SPEND_201_PRIME_EX_TAU,58,0,0
SHARE_CE_SPEND_279_FINANCIAL_AGGREGATE,208,0,0
SHARE_CE_SPEND_279_FINANCIAL_PRIME_ENRICHED,208,0,0


## 3. Classify each object

**Column roles**, by name, first match wins:
- *Transaction id*: `EARNEST_TRANSACTION_ID`, `TRANSID`, `UNIQUE_CARD_TRANSACTION_ID`, `UNIQUE_BANK_TRANSACTION_ID`, `FDTXNKEY` (Fable), `PARTNER_TRANSACTION_ID`, `PARTNER_TRANSID`, `TRANSACTION_ID`, `KLARNA_TXN_ID`.
- *Description*: `BILLING_DESCRIPTION` (279), `DESCRIPTION` (Yodlee, Fable), `MERCHDESC` (Prime Omega merchant table), `EARNEST_DESCRIPTION`, `CLEANED_DESCRIPTION`. Counted only when its table is row-level or a merchant table, since aggregates use `DESCRIPTION` for other things.
- *Main date*: a transaction date where there is one (`TRANSACTION_DATE`, `TRANS_DATE`, `TXNDATE`, ...), otherwise the period date of an aggregate, otherwise the first date column.
- *File date* (used only to prune the December query): `PANEL_FILE_CREATED_DATE`, `FILE_CREATE_DATE`, `FILEDATE`, `FILE_CREATED_DATE`. A transaction never lands in a file created before it, so `file_date >= 2025-11-24` keeps every December transaction.

**Grain**: `row-level` (has a transaction id), `merchant` (merchant/descriptor dimension), `aggregate` (has a date, no transaction id), `reference` (no date), `delivery metadata` (`DATADELIVERED*`, `MANIFEST`, trackers) or `external` (S3-backed `_EXT` copy of a base table).

**Variant** of a row-level table: `current`, `snapshot`, `change log`, or `backup / scratch`.

**Mirrors.** An object whose (schema, name, row_count, bytes) equals one in a higher-priority database is marked `mirror_of` that object and not probed. For views (no row counts) the column list must match instead.

**Access.** Some shares list every schema but grant this role read access to only some of them; the TRANSACT_USA and 2822 databases split their schemas across sibling databases this way. `readable` records whether this role can read the schema, and a mirror always points at a readable copy. Unreadable objects with no readable copy are listed but not probed.

In [4]:
TXN_ID = ["EARNEST_TRANSACTION_ID", "TRANSID", "UNIQUE_CARD_TRANSACTION_ID", "UNIQUE_BANK_TRANSACTION_ID", "FDTXNKEY",
          "PARTNER_TRANSACTION_ID", "PARTNER_TRANSID", "TRANSACTION_ID", "KLARNA_TXN_ID"]
DESC = ["BILLING_DESCRIPTION", "DESCRIPTION", "MERCHDESC", "EARNEST_DESCRIPTION", "CLEANED_DESCRIPTION"]
TXN_DATE = ["TRANSACTION_DATE", "TRANS_DATE", "TXNDATE", "PARTNER_TRANS_DATE", "POST_DATE", "POSTEDDATE", "PARTNER_POST_DATE"]
PERIOD_DATE = ["DATE", "OPTIMIZED_DATE", "PERIOD_START_DT", "PERIOD_START_DATE", "EARNEST_WEEK_BEGIN_DATE", "PERIOD_END_DT",
               "PERIOD_END_DATE", "EARNEST_WEEK_END_DATE", "QUARTER_START_DATE", "REPORT_START_DATE", "START_DATE",
               "START_MONTH", "COHORT_MONTH", "COHORT_TIME_PERIOD", "REPORTDATE", "PARTITION_DATE", "PANEL_START_DATE",
               "VERSION_DATE", "DELIVERY_DATE", "DELIVERY_TIME", "FILE_CREATE_DATE", "FILEDATE", "FILE_CREATED_DATE",
               "PANEL_FILE_CREATED_DATE", "APIPULLDATE"]
FILE_DATE = ["PANEL_FILE_CREATED_DATE", "FILE_CREATE_DATE", "FILEDATE", "FILE_CREATED_DATE"]
DATE_TYPES = {"DATE", "TIMESTAMP_NTZ", "TIMESTAMP_LTZ", "TIMESTAMP_TZ"}
NOT_A_DATA_DATE = {"LAST_ALTERED", "CHANGE_DATE", "UPDATEDATE", "PATCHEDDATE", "ARCHIVEDDATE", "VERSION"}

by_obj = cols.groupby("fq")
col_names = by_obj.column_name.apply(list)
date_cols = cols[cols.data_type.isin(DATE_TYPES)].groupby("fq").column_name.apply(list)

def first(cands, present):
    return next((c for c in cands if c in present), None)

def roles(fq):
    names = col_names.get(fq, [])
    dates = date_cols.get(fq, [])
    txn_date = first(TXN_DATE, dates)
    main_date = txn_date or first(PERIOD_DATE, dates) or next((d for d in dates if d not in NOT_A_DATA_DATE), None) \
        or (dates[0] if dates else None)
    return pd.Series({
        "txn_id_col": first(TXN_ID, names),
        "desc_col_any": first(DESC, names),
        "txn_date_col": txn_date,
        "main_date_col": main_date,
        "file_date_col": first(FILE_DATE, dates),
        "n_cols": len(names),
        "col_sig": "|".join(sorted(names)),
    })

objs = objs.join(objs.fq.apply(roles))

META_RX = r"^(DATADELIVERED|MANIFEST|SYNC_TRACKER|TABLE_INFORMATION)"
def grain(r):
    if r.table_type == "EXTERNAL TABLE":
        return "external"
    if re.match(META_RX, r.table_name):
        return "delivery metadata"
    if r.txn_id_col:
        return "row-level"
    if r.table_name.startswith("MID_V") or r.desc_col_any == "MERCHDESC":
        return "merchant"
    if r.main_date_col:
        return "aggregate"
    return "reference"

def variant(r):
    s, n = r.table_schema, r.table_name
    if s.endswith("_BKP") or re.search(r"(BKP|_OLD|_TEMP|_TEST|_TMP|^WAYFAIR_TEST)", n):
        return "backup / scratch"
    if "INCREMENTAL" in s or re.search(r"(_LOG_|_LOG$|CHANGESET|INCREMENTALS)", n):
        return "change log"
    if "SNAPSHOT" in s or "SNAPSHOT" in n or "ARCHIVE" in s or "ARCHIVE" in n:
        return "snapshot / archive"
    return "current"

objs["grain"] = objs.apply(grain, axis=1)
objs["variant"] = objs.apply(lambda r: variant(r) if r.grain == "row-level" else None, axis=1)
objs["has_txn_description"] = objs.desc_col_any.notna() & objs.grain.isin(["row-level", "merchant"])
objs["txn_description_col"] = objs.desc_col_any.where(objs.has_txn_description)

# Mirrors: same schema/name and identical size (tables) or identical columns (views) in a higher-priority database.
PRIORITY = ["SHARE_EARNEST_279", "SHARE_CE_SPEND_279_FINANCIAL_PRIME_ENRICHED", "SHARE_CE_SPEND_279_FINANCIAL_PRIME_ENRICHED_TAU",
            "SHARE_CE_SPEND_279_FINANCIAL_AGGREGATE", "SHARE_CE_PRIME_ENRICHED_OMEGA_2822", "SHARE_CE_PRIME_ENRICHED_TAU_2822",
            "SHARE_FABLE", "SHARE_FABLE_DAILY_T7_V2", "SHARE_FABLE_DAILY_T7",
            "SHARE_CE_TRANSACT_USA_COHORT", "SHARE_CE_TRANSACT_USA_DAILY_SIGNAL", "SHARE_CE_TRANSACT_USA_GROUND_TRUTH"]
objs["db_rank"] = objs.table_catalog.map({d: i for i, d in enumerate(PRIORITY)}).fillna(-1)
objs["mirror_key"] = objs.apply(lambda r: (r.table_schema, r.table_name, r.table_type,
                                           (r.row_count, r.bytes) if pd.notna(r.row_count) and r.table_type != "VIEW" else r.col_sig), axis=1)
# A share can list every schema in INFORMATION_SCHEMA yet grant USAGE on only some (the TRANSACT_USA trio and
# the two 2822 databases split their schemas this way), so a mirror must point at a copy we can read.
def schema_readable(db, schema):
    try:
        conn.cursor().execute(f'SHOW TERSE TABLES IN SCHEMA {db}."{schema}" LIMIT 1')
        return True
    except Exception:
        return False

pairs = objs[["table_catalog", "table_schema"]].drop_duplicates()
pairs["readable"] = [schema_readable(d, s) for d, s in pairs.itertuples(index=False)]
objs = objs.merge(pairs, on=["table_catalog", "table_schema"], how="left")

ranked = objs[(objs.db_rank >= 0) & objs.readable].sort_values("db_rank")
canon = ranked.drop_duplicates("mirror_key").set_index("mirror_key").fq
objs["mirror_of"] = objs.apply(lambda r: canon.get(r.mirror_key) if r.db_rank >= 0 and canon.get(r.mirror_key) not in (None, r.fq) else None, axis=1)
print("schemas not readable by this role:")
print(pairs[~pairs.readable].groupby("table_catalog").table_schema.apply(list).to_string())

objs.drop(columns=["col_sig", "mirror_key"]).to_csv(os.path.join(RES, "objects.csv"), index=False)
summary = objs.groupby(["table_catalog", "grain"]).size().unstack(fill_value=0)
summary["mirrored"] = objs.groupby("table_catalog").mirror_of.count()
summary

schemas not readable by this role:
table_catalog
SHARE_CE_PRIME_ENRICHED_OMEGA_2822                                         [PRIME_TAU, PRIME_INCREMENTAL_TAU, PRIME_SNAPSHOTS_TAU]
SHARE_CE_PRIME_ENRICHED_TAU_2822                   [PRIME_INCREMENTAL_OMEGA, PRIME_OMEGA, PRIME_OMEGA_STATIC, PRIME_SNAPSHOTS_O...
SHARE_CE_SPEND_279_FINANCIAL_AGGREGATE             [DIRECT_INCREMENTAL, DIRECT_INCREMENTAL_SIGMA, DIRECT_SIGMA, DIRECT_SNAPSHOT...
SHARE_CE_SPEND_279_FINANCIAL_PRIME_ENRICHED        [ARCHIVE, DIRECT_INCREMENTAL_TAU, DIRECT_TAU, KPI, STATIC, IRS_PAYMENTS, QUE...
SHARE_CE_SPEND_279_FINANCIAL_PRIME_ENRICHED_TAU    [ARCHIVE, DIRECT, DIRECT_INCREMENTAL, DIRECT_INCREMENTAL_SIGMA, DIRECT_SNAPS...
SHARE_CE_TRANSACT_USA_COHORT                       [CE_1315_DAILY_SIGNAL, COMBINED_USA1_USA2_DAILY_SIGNAL, USA1_DAILY_SIGNAL, U...
SHARE_CE_TRANSACT_USA_DAILY_SIGNAL                      [CE_1315_COHORT, COMBINED_USA1_USA2_COHORT, USA1_COHORT, USA_GROUND_TRUTH]
SHARE_CE_TRANSACT_USA_GROUND_TRUTH

grain,aggregate,delivery metadata,external,merchant,reference,row-level,mirrored
table_catalog,,,,,,,
EARNEST,142,1,0,0,10,134,0
FABLE,14,1,0,0,4,0,0
SHARE_CE_1315_DAILY_SIGNAL,31,13,0,0,8,0,0
SHARE_CE_1415_DAILY_SIGNAL,21,6,0,0,4,0,0
SHARE_CE_PRIME_ENRICHED_OMEGA_2822,5,22,0,2,8,6,15
SHARE_CE_PRIME_ENRICHED_TAU_2822,5,22,0,2,8,6,28
SHARE_CE_SPEND_201_PRIME_EX_TAU,12,31,0,0,5,10,0
SHARE_CE_SPEND_279_FINANCIAL_AGGREGATE,96,55,0,0,39,18,208
SHARE_CE_SPEND_279_FINANCIAL_PRIME_ENRICHED,96,55,0,0,39,18,208


## 4. Data probes

Two kinds of query per probed object, run in parallel and cached in `results/probes.csv`:
- **minmax**: `MIN` / `MAX` of the main date column. On a base table Snowflake usually answers this from metadata without scanning. If the result falls outside 1990 to tomorrow (placeholder dates), a second guarded query returns the range within those bounds.
- **dec25**: `COUNT(DISTINCT txn_id)` for rows whose transaction date is in December 2025 (row-level tables with a transaction date only), pruned on the file date where there is one.

Not probed: mirrors, external tables and delivery metadata.

In [5]:
PROBE_CSV = os.path.join(RES, "probes.csv")
done = pd.read_csv(PROBE_CSV) if os.path.exists(PROBE_CSV) else pd.DataFrame(
    columns=["fq", "kind", "status", "v1", "v2", "seconds", "run_at", "sql"])
keep = done.status.eq("ok") if RETRY_FAILED else done.status.notna()
skip = set(zip(done[keep].fq, done[keep].kind))
done = done[keep]

def ident(fq):
    db, sc, tb = fq.split(".", 2)
    return f'{db}."{sc}"."{tb}"'

def jobs():
    probed = objs[objs.mirror_of.isna() & objs.readable & ~objs.grain.isin(["external", "delivery metadata"])]
    for r in probed.itertuples():
        t = ident(r.fq)
        if r.main_date_col and (r.fq, "minmax") not in skip:
            yield r.fq, "minmax", f'SELECT MIN("{r.main_date_col}")::DATE, MAX("{r.main_date_col}")::DATE FROM {t}', MINMAX_TIMEOUT_S
        if r.grain == "row-level" and r.txn_date_col and (r.fq, "dec25") not in skip:
            prune = f' AND "{r.file_date_col}" >= \'2025-11-24\'' if r.file_date_col else ""
            yield r.fq, "dec25", (f'SELECT COUNT(DISTINCT "{r.txn_id_col}"), COUNT(*) FROM {t} '
                                  f'WHERE "{r.txn_date_col}" >= \'{DEC_FROM}\' AND "{r.txn_date_col}" < \'{DEC_TO}\'{prune}'), DEC_TIMEOUT_S

_local = threading.local()
_lock = threading.Lock()

def cursor():
    if not hasattr(_local, "conn"):
        _local.conn = snowflake.connector.connect(connection_name=CONN_NAME, **CONN_KW)
    return _local.conn.cursor()

def guarded(fq, col, cur):
    sql = (f'SELECT MIN("{col}")::DATE, MAX("{col}")::DATE FROM {ident(fq)} '
           f'WHERE "{col}" >= \'1990-01-01\' AND "{col}" < DATEADD(day, 2, CURRENT_DATE)')
    return cur.execute(sql).fetchone(), sql

def run(job):
    fq, kind, sql, timeout = job
    t0 = time.time()
    cur = cursor()
    try:
        cur.execute(f"ALTER SESSION SET STATEMENT_TIMEOUT_IN_SECONDS = {timeout}")
        v1, v2 = cur.execute(sql).fetchone()
        if kind == "minmax" and v1 is not None and (v1 < dt.date(1990, 1, 1) or v2 > TODAY + dt.timedelta(days=1)):
            col = objs.loc[objs.fq == fq, "main_date_col"].iat[0]
            (v1, v2), sql = guarded(fq, col, cur)
            kind_status = "ok (outliers excluded)"
        else:
            kind_status = "ok"
        status = kind_status
    except Exception as e:
        msg = str(e)
        status = "timeout" if "timeout" in msg.lower() or "canceled" in msg.lower() else "error: " + msg.splitlines()[-1][:200]
        v1 = v2 = None
    row = dict(fq=fq, kind=kind, status=status, v1=v1, v2=v2, seconds=round(time.time() - t0, 1),
               run_at=dt.datetime.now().isoformat(timespec="seconds"), sql=sql)
    with _lock:
        pd.DataFrame([row]).to_csv(PROBE_CSV, mode="a", header=not os.path.exists(PROBE_CSV), index=False)
    return row

todo = sorted(jobs(), key=lambda j: (j[1] == "dec25", j[0]))   # all cheap minmax first
print(f"{len(todo)} queries to run ({sum(j[1] == 'dec25' for j in todo)} December counts)")
if DRY_RUN:
    todo = []
t0 = time.time()
with ThreadPoolExecutor(THREADS) as ex:
    for i, f in enumerate(as_completed([ex.submit(run, j) for j in todo]), 1):
        r = f.result()
        if r["kind"] == "dec25" or not r["status"].startswith("ok") or i % 200 == 0:
            print(f"[{i}/{len(todo)} {time.time() - t0:,.0f}s] {r['kind']:6} {r['status'][:40]:40} {r['seconds']:>7}s {r['fq']}")

0 queries to run (0 December counts)


## 5. Results

`probes` holds the latest result of each probe. `inv` is the full inventory: one row per object, mirrors carrying the probe results of the object they copy.

In [6]:
probes = pd.read_csv(PROBE_CSV).sort_values("run_at").drop_duplicates(["fq", "kind"], keep="last")
p = probes.pivot(index="fq", columns="kind", values=["status", "v1", "v2"])
p.columns = [f"{a}_{b}" for a, b in p.columns]
p = p.rename(columns={"v1_minmax": "data_from", "v2_minmax": "data_to", "v1_dec25": "dec25_distinct_txns",
                      "v2_dec25": "dec25_rows", "status_minmax": "minmax_status", "status_dec25": "dec25_status"})
for c in ["data_from", "data_to", "dec25_distinct_txns", "dec25_rows", "minmax_status", "dec25_status"]:
    if c not in p:
        p[c] = None

inv = objs.copy()
inv["probe_source"] = inv.mirror_of.fillna(inv.fq)
inv = inv.merge(p, left_on="probe_source", right_index=True, how="left")
inv = inv.merge(dbs[["name", "provider"]], left_on="table_catalog", right_on="name", how="left").drop(columns="name")
inv["data_from"] = pd.to_datetime(inv.data_from)
inv["data_to"] = pd.to_datetime(inv.data_to)
inv["days_stale"] = (pd.Timestamp(TODAY) - pd.to_datetime(inv.data_to)).dt.days
inv["dec25_distinct_txns"] = pd.to_numeric(inv.dec25_distinct_txns)
inv["dec25_rows"] = pd.to_numeric(inv.dec25_rows)
inv["gb"] = (inv.bytes / 1e9).round(1)

OUT_COLS = ["provider", "table_catalog", "table_schema", "table_name", "table_type", "grain", "variant", "readable", "mirror_of",
            "has_txn_description", "txn_description_col", "main_date_col", "data_from", "data_to", "days_stale",
            "dec25_distinct_txns", "dec25_rows", "txn_id_col", "row_count", "gb", "created", "last_altered",
            "clustering_key", "minmax_status", "dec25_status"]
inv = inv[OUT_COLS].sort_values(["provider", "table_catalog", "table_schema", "table_name"]).reset_index(drop=True)
inv.to_csv(os.path.join(RES, "inventory.csv"), index=False)

# Excel copy for browsing (timezones stripped; Excel cannot store them)
xl = inv.copy()
for c in ["created", "last_altered"]:
    xl[c] = pd.to_datetime(xl[c], utc=True).dt.tz_localize(None)
with pd.ExcelWriter(os.path.join(RES, "inventory.xlsx")) as w:
    xl[xl.grain == "row-level"].to_excel(w, sheet_name="row_level", index=False)
    xl.to_excel(w, sheet_name="all_objects", index=False)
    dbs.assign(created_on=pd.to_datetime(dbs.created_on, utc=True).dt.tz_localize(None)).to_excel(w, sheet_name="databases", index=False)
print(inv.minmax_status.value_counts(dropna=False), inv.dec25_status.value_counts(dropna=False), sep="\n\n")

minmax_status
ok                                                                                                                                                                                                                 1841
NaN                                                                                                                                                                                                                1520
ok (outliers excluded)                                                                                                                                                                                               39
error: Object 'SHARE_FABLE.CLIENT_VIEW.SIGNAL_DAILY_V2_SPOTIFY_PATCH' does not exist or not authorized. Your primary role DATA_TEAM or one of your secondary roles must have at least one privilege granted on        3
ok (derived from base tables)                                                                                             

### 5a. Databases

In [7]:
db_view = (inv.groupby("table_catalog")
             .agg(objects=("table_name", "size"),
                  row_level=("grain", lambda g: (g == "row-level").sum()),
                  with_description=("has_txn_description", "sum"),
                  mirrored=("mirror_of", "count"),
                  readable=("readable", "sum"),
                  data_from=("data_from", "min"), data_to=("data_to", "max"),
                  gb=("gb", "sum")))
db_view = dbs.set_index("name")[["provider", "kind", "origin", "available"]].join(db_view, how="left")
db_view.sort_values(["provider"])

,provider,kind,origin,available,objects,row_level,with_description,mirrored,readable,data_from,data_to,gb
name,,,,,,,,,,,,
SHARE_CE_TRANSACT_USA_GROUND_TRUTH,Consumer Edge Transact USA: aggregates,IMPORTED DATABASE,DMOEKXS.WIB77105.CE_TRANSACT_USA_GROUND_TRUTH,True,638.0,0.0,0.0,611.0,27.0,2012-07-01,2026-10-14,121419.5
SHARE_CE_TRANSACT_USA_DAILY_SIGNAL,Consumer Edge Transact USA: aggregates,IMPORTED DATABASE,DMOEKXS.WIB77105.CE_TRANSACT_USA_DAILY_SIGNAL,True,638.0,0.0,0.0,95.0,543.0,2012-07-01,2026-10-14,121419.5
SHARE_CE_TRANSACT_USA_COHORT,Consumer Edge Transact USA: aggregates,IMPORTED DATABASE,DMOEKXS.WIB77105.CE_TRANSACT_USA_COHORT,True,638.0,0.0,0.0,570.0,68.0,2012-07-01,2026-10-14,121419.5
SHARE_CE_1315_DAILY_SIGNAL,Consumer Edge Transact: daily signal (1315 / 1415),IMPORTED DATABASE,XW17098.PW55959.CE_1315_DAILY_SIGNAL,True,52.0,0.0,0.0,0.0,52.0,2012-07-01,2026-10-14,234.0
SHARE_CE_1415_DAILY_SIGNAL,Consumer Edge Transact: daily signal (1315 / 1415),IMPORTED DATABASE,XW17098.PW55959.CE_1415_DAILY_SIGNAL,True,31.0,0.0,0.0,0.0,31.0,2012-07-01,2026-10-14,146.2
SHARE_CE_SPEND_279_FINANCIAL_PRIME_ENRICHED,Consumer Edge: Earnest spend 279 (legacy schema),IMPORTED DATABASE,XW17098.PW55959.SPEND_279_FINANCIAL_PRIME_ENRICHED,True,208.0,18.0,17.0,208.0,72.0,2011-11-24,2026-10-14,28777.8
SHARE_CE_SPEND_279_FINANCIAL_PRIME_ENRICHED_TAU,Consumer Edge: Earnest spend 279 (legacy schema),IMPORTED DATABASE,XW17098.PW55959.SPEND_279_FINANCIAL_PRIME_ENRICHED_TAU,True,208.0,18.0,17.0,208.0,35.0,2011-11-24,2026-10-14,28777.8
SHARE_CE_SPEND_279_FINANCIAL_AGGREGATE,Consumer Edge: Earnest spend 279 (legacy schema),IMPORTED DATABASE,XW17098.PW55959.SPEND_279_FINANCIAL_AGGREGATE,True,208.0,18.0,17.0,208.0,92.0,2011-11-24,2026-10-14,28777.8
SHARE_CE_PRIME_ENRICHED_OMEGA_2822,"Consumer Edge: Prime Enriched, new Earnest schema (2822)",IMPORTED DATABASE,XW17098.PW55959.PRIME_ENRICHED_OMEGA_2822,True,43.0,6.0,2.0,15.0,28.0,2000-12-30,2026-10-13,31169.4


### 5b. Row-level transaction tables (originals, not mirrors)

The candidates for transaction-level work. December 2025 counts compare panel sizes; `dec25_rows > dec25_distinct_txns` means a transaction repeats (for example 279 repeats a transaction once per merchant tag).

In [8]:
rl = inv[(inv.grain == "row-level") & inv.mirror_of.isna()]
rl[["provider", "table_catalog", "table_schema", "table_name", "table_type", "variant", "has_txn_description",
    "txn_description_col", "data_from", "data_to", "days_stale", "dec25_distinct_txns", "dec25_rows", "gb", "dec25_status"]
  ].sort_values(["provider", "variant", "dec25_distinct_txns"], ascending=[True, True, False])

,provider,table_catalog,table_schema,table_name,table_type,variant,has_txn_description,txn_description_col,data_from,data_to,days_stale,dec25_distinct_txns,dec25_rows,gb,dec25_status
2631,"Consumer Edge: Prime Enriched, new Earnest schema (2822)",SHARE_CE_PRIME_ENRICHED_OMEGA_2822,PRIME_INCREMENTAL_OMEGA,TRANS_LOG_V1_0,BASE TABLE,change log,False,NaN,2005-03-16,2026-10-13,2.0,2.143129e+08,2.143129e+08,877.7,ok
2678,"Consumer Edge: Prime Enriched, new Earnest schema (2822)",SHARE_CE_PRIME_ENRICHED_TAU_2822,PRIME_INCREMENTAL_TAU,TRANS_LOG_V1_0,BASE TABLE,change log,False,NaN,2013-01-01,2026-10-13,2.0,1.688271e+06,1.688271e+06,211.7,ok
2646,"Consumer Edge: Prime Enriched, new Earnest schema (2822)",SHARE_CE_PRIME_ENRICHED_OMEGA_2822,PRIME_OMEGA,TRANS_V1_0,BASE TABLE,current,False,NaN,2000-12-30,2026-10-13,2.0,1.938900e+09,1.938900e+09,12933.9,ok
2706,"Consumer Edge: Prime Enriched, new Earnest schema (2822)",SHARE_CE_PRIME_ENRICHED_TAU_2822,PRIME_TAU,TRANS_V1_0,BASE TABLE,current,False,NaN,2013-01-01,2026-10-13,2.0,8.216834e+08,8.216834e+08,1880.9,ok
2652,"Consumer Edge: Prime Enriched, new Earnest schema (2822)",SHARE_CE_PRIME_ENRICHED_OMEGA_2822,PRIME_SNAPSHOTS_OMEGA,TRANS_V1_0,BASE TABLE,snapshot / archive,False,NaN,2000-12-30,2026-10-12,3.0,1.938317e+09,1.938317e+09,12973.6,ok
2699,"Consumer Edge: Prime Enriched, new Earnest schema (2822)",SHARE_CE_PRIME_ENRICHED_TAU_2822,PRIME_SNAPSHOTS_TAU,TRANS_V1_0,BASE TABLE,snapshot / archive,False,NaN,2013-01-01,2026-10-12,3.0,8.216643e+08,8.216643e+08,1805.0,ok
2725,Consumer Edge: Yodlee card & bank feeds (201),SHARE_CE_SPEND_201_PRIME_EX_TAU,FEED3_PANELS,FEED3_BANK_PANELS_V1_0,BASE TABLE,current,True,DESCRIPTION,2005-03-16,2026-10-14,1.0,7.592392e+08,7.592392e+08,3965.9,ok
2755,Consumer Edge: Yodlee card & bank feeds (201),SHARE_CE_SPEND_201_PRIME_EX_TAU,FEED6_PANELS,FEED6_BANK_PANELS_V1_0,BASE TABLE,current,True,DESCRIPTION,2001-06-24,2026-10-16,-1.0,2.462519e+08,2.462519e+08,3536.9,ok
2726,Consumer Edge: Yodlee card & bank feeds (201),SHARE_CE_SPEND_201_PRIME_EX_TAU,FEED3_PANELS,FEED3_CARD_PANELS_V1_0,BASE TABLE,current,True,DESCRIPTION,2001-01-01,2026-10-15,0.0,3.633731e+07,3.633731e+07,212.4,ok
2740,Consumer Edge: Yodlee card & bank feeds (201),SHARE_CE_SPEND_201_PRIME_EX_TAU,FEED4_PANELS,FEED4_BANK_PANELS_V1_0,BASE TABLE,current,True,DESCRIPTION,2005-03-16,2026-10-14,1.0,2.904135e+07,2.904135e+07,1684.4,ok


### 5c. Everything else (aggregates, merchant and reference tables), originals only

In [9]:
other = inv[(inv.grain != "row-level") & inv.mirror_of.isna() & (inv.grain != "external")]
other[["provider", "table_catalog", "table_schema", "table_name", "table_type", "grain", "has_txn_description",
       "main_date_col", "data_from", "data_to", "days_stale", "row_count", "gb", "minmax_status"]]

,provider,table_catalog,table_schema,table_name,table_type,grain,has_txn_description,main_date_col,data_from,data_to,days_stale,row_count,gb,minmax_status
0,Consumer Edge Transact USA: aggregates,SHARE_CE_TRANSACT_USA_COHORT,CE_1315_COHORT,PERIOD_BRAND_CROSSSHOP,BASE TABLE,aggregate,False,PERIOD_START_DT,2022-10-01,2026-09-01,44.0,79076344.0,1.5,ok
2,Consumer Edge Transact USA: aggregates,SHARE_CE_TRANSACT_USA_COHORT,CE_1315_COHORT,PERIOD_BRAND_DEMO,BASE TABLE,aggregate,False,PERIOD_START_DT,2022-10-01,2026-09-01,44.0,56269933.0,1.5,ok
4,Consumer Edge Transact USA: aggregates,SHARE_CE_TRANSACT_USA_COHORT,CE_1315_COHORT,PERIOD_BRAND_FIRSTSHOP,BASE TABLE,aggregate,False,PERIOD_START_DT,2022-10-01,2026-09-01,44.0,19541686.0,0.3,ok
6,Consumer Edge Transact USA: aggregates,SHARE_CE_TRANSACT_USA_COHORT,CE_1315_COHORT,PERIOD_BRAND_GEO,BASE TABLE,aggregate,False,PERIOD_START_DT,2022-10-01,2026-09-01,44.0,180755102.0,4.1,ok
8,Consumer Edge Transact USA: aggregates,SHARE_CE_TRANSACT_USA_COHORT,CE_1315_COHORT,PERIOD_BRAND_TICKET_BUCKET,BASE TABLE,aggregate,False,PERIOD_START_DT,2022-10-01,2026-09-01,44.0,25553713.0,0.4,ok
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3399,SRS internal: Fable-derived aggregates,FABLE,COMMON,SIGNAL_DAILY_T7EU,BASE TABLE,aggregate,False,SHIPPED_DATE,2016-01-07,2025-09-11,399.0,361424904.0,26.4,ok
3400,SRS internal: Fable-derived aggregates,FABLE,COMMON,TRIAL_PANELS_DATA_OLD,BASE TABLE,aggregate,False,REPORT_DATE,2022-04-22,2022-07-20,1548.0,29624392.0,1.0,ok
3401,SRS internal: Fable-derived aggregates,FABLE,COMMON,TRIAL_PANELS_METRICS_OLD,BASE TABLE,aggregate,False,START_DATE,2015-01-01,2022-02-27,1691.0,6693.0,0.0,ok
3402,SRS internal: Fable-derived aggregates,FABLE,COMMON,TRIAL_PANELS_SETTINGS_OLD,BASE TABLE,reference,False,None,NaT,NaT,NaN,24074.0,0.0,NaN


## Status

_Run completed 2026-10-15. Small warehouse for metadata and most probes, LARGE for the slow December counts._

**Coverage**
- 23 databases match the prefixes. 18 are reachable, giving 3,404 tables / views.
- Not reachable:
  - Four healthcare-claims shares and `SHARE_EARNEST_RETAIL_PRICING` have been revoked.
  - `SHARE_EARNEST_327`, `CALLOUTS` and `FOOT_TRAFFIC` errored.
  - `SHARE_FABLE.CLIENT_VIEW.SIGNAL_DAILY_V2_SPOTIFY_PATCH` is not authorised for DATA_TEAM.
- About 1,400 objects are exact mirrors and are not counted separately. Their `mirror_of` column points to the copy that was probed:
  - the four `*_279` databases are mirrors of `SHARE_EARNEST_279`;
  - `TAU_2822` mirrors `OMEGA_2822`;
  - the three Fable databases are identical;
  - the three `TRANSACT_USA` databases are identical.
- Every December 2025 count completed.
- `EARNEST.COMMON.CE_PRIME_OMEGA_PANEL_TXNS`: its own MIN/MAX would not finish, so the date range was derived from `TRANS_V1_0`. The derivation keeps the view's panel join, its 2019-01-01 floor and its affected-accounts exclusion. All its other joins are LEFT JOINs.

**Row-level transaction sources, December 2025 distinct transactions**

| Source | Object | Descriptor | Data from → to | Dec-25 txns |
|---|---|---|---|---:|
| Prime Omega | `OMEGA_2822.PRIME_OMEGA.TRANS_V1_0` | merchant-level only, via `MID_V1_0.MERCHDESC` | 2000 → 2026-10-13 | 1,939M |
| Prime Omega (Tau panel) | `TAU_2822.PRIME_TAU.TRANS_V1_0` | merchant-level only | 2013 → 2026-10-13 | 822M |
| Yodlee feed 3 bank | `SPEND_201_PRIME_EX_TAU.FEED3_PANELS.FEED3_BANK_PANELS_V1_0` | `DESCRIPTION` | 2005 → 2026-10-14 | 759M |
| 279 Tau | `SHARE_EARNEST_279.PRIME_TAU.SPEND_279_TRANSACTIONS_V3_0` | `BILLING_DESCRIPTION` | 2023-03-28 → 2026-10-14 | 432M |
| Josh's view (CS 2019 / 2022 panels) | `EARNEST.COMMON.CE_PRIME_OMEGA_PANEL_TXNS` | `MERCHDESC` (merchant-level) | 2019-01-01 → 2026-10-14 | 418M |
| 279 Prime | `SHARE_EARNEST_279.PRIME.SPEND_279_TRANSACTIONS_V3_0` | `BILLING_DESCRIPTION` | 2015-02-26 → 2026-10-14 | 278M |
| Yodlee feed 6 bank | `FEED6_BANK_PANELS_V1_0` | `DESCRIPTION` | 2001 → 2026-10-16 | 246M |
| Fable FR v3 / FR v2 / UK / ES / IT / DE | `SHARE_FABLE.FABLE_SIGNAL.*_TXN` | `DESCRIPTION` | → 2026-10-12/14 | 71 / 55 / 53 / 26 / 13 / 11M |
| Yodlee feed 3 / 4 / 6 card | `FEED{3,4,6}_CARD_PANELS_V1_0` | `DESCRIPTION` | → 2026-10-14/15 | 36 / 27 / 18M |
| Yodlee feed 4 bank | `FEED4_BANK_PANELS_V1_0` | `DESCRIPTION` | → 2026-10-14 | 29M |
| Earnest ticker extracts | `EARNEST.<TICKER>.TXNS` (PDD, UBER, DASH, NFLX, …) | `BILLING_DESCRIPTION` | → 2026-10-13/14 | 0.1–40M each |

**Early observations, for the follow-up overlap analysis**
- Several tables are just versions of one dataset:
  - `PRIME_SNAPSHOTS*` tables match their current table to within 0.1% for December.
  - The 279 `PRIME_INCREMENTAL*` change logs match to within 0.5%.
  - The Prime Omega `PRIME_INCREMENTAL_*` logs are much smaller (214M and 1.7M), so they hold only the changed rows.
  - Fable `*_SIGNAL` views equal their `*_TXN` tables.
  - Use the current table unless point-in-time history is needed.
- Some sources have stopped:
  - the 279 `DIRECT*` and `SIGMA` tables;
  - Yodlee feeds 1 and 2, which ended September 2025.
  - None of these has any December 2025 data.
- Prime Omega `TRANS_V1_0` is the largest US source, but has no per-transaction descriptor. The descriptor is per merchant (`MERCHID` → `MID_V1_0.MERCHDESC`).
- Per-transaction descriptors come from three places: 279 (`BILLING_DESCRIPTION`), Yodlee (`DESCRIPTION`) and Fable (`DESCRIPTION`).
- 279 `PRIME` has about 1.4 rows per distinct transaction (390M rows vs 278M ids), so deduplicate on the transaction id.
- 279 `PRIME_TAU` is larger than `PRIME` (432M vs 278M), but only starts on 2023-03-28.
- Josh's view restricts Prime Omega to the two consistent-shopper panels: 418M of the 1,939M December transactions, about 22%.

**Next:** a subset and overlap notebook to find out:
- whether 279 PRIME / PRIME_TAU are subsets of Prime Omega / Tau;
- how they overlap with Yodlee feeds 3, 4 and 6;
- whether per-merchant `MERCHDESC` is good enough to identify agentic (`LINKAGNT*`) transactions, compared with per-transaction descriptors.